# GLM Tuning-Variable Analysis (PFC, mFC dataset)

Port of the LEC GLM analysis pipeline (`code/LEC_glm_tuning_variables.ipynb`) to
the mPFC dataset under `mFC_data/data/`. Same model, same plots, same variants
— with two differences:

1. **No head direction.** The mFC dataset has no `HD_raw` recording, so HD is
   dropped from every fit via `regressors_to_include`. 8 regressors instead of
   9; default-bin design matrix is 91 cols (127 − 36 HD).
2. **Disk-loaded data.** The mFC dataset is a directory of `.npy` files rather
   than a pickle. `build_data_dic_from_pfc(data_folder, recdays)` (defined in
   the PFC sibling module) constructs the same nested-dict shape that
   `run_glm_analysis` consumes.

The substantive analyses (F-test significance, CPD / R² / ΔR², joint time-any
and distance-any groups, raised-cosine variant, 20-bin GP column-matched
variant, reference-coded sanity check) are unchanged from the LEC notebook.

## Method: per-regressor significance via permutation F-test

For each neuron, we fit a per-neuron OLS GLM:

    y_t = X_t β + ε_t

`X` is the design matrix of one-hot encoded task variables. PFC variables:
place (21 bins), goal progress (10 equal-width bins), speed, acceleration,
time-since-reward, time-to-reward, distance-since-reward, distance-to-reward
(10 decile bins each). **HD is dropped** (no recording). Total: **91 cols**
at the default `gp_n_bins=10`, or 101 cols at `gp_n_bins=20`.

To test whether a regressor group (e.g., all 21 place columns) explains
firing-rate variance, we use the nested F-test against a permutation null
(circularly shifted firing rate). A neuron is "tuned" if its observed F
exceeds the 95th percentile of the 100-shift null.

CPD = (RSS_reduced − RSS_full)/RSS_reduced is reported per regressor as an
effect-size complement to the F-test.

In [1]:
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt
import os, sys
from tqdm import tqdm
from importlib import reload
from collections import defaultdict
import glm_analysis_v2
reload(glm_analysis_v2)

<module 'glm_analysis_v2' from '/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/code/glm_analysis_v2.py'>

In [2]:
DATA_FOLDER = '/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/data'
META = os.path.join(DATA_FOLDER, 'MetaData')

In [3]:
# Canonical recday list (25 double-day recordings, ABCD tasks only)
mouse_recdays = list(np.load(os.path.join(META, 'combined_ABCDonly_days.npy')))
mouse_recdays = [str(mr) for mr in mouse_recdays]
print(f'{len(mouse_recdays)} recdays in combined_ABCDonly_days.npy')
print('first 3:', mouse_recdays[:3])

25 recdays in combined_ABCDonly_days.npy
first 3: ['ab03_01092023_02092023', 'ab03_05092023_06092023', 'ab03_29082023_30082023']


In [4]:
# Memory note: eager-loading all 25 recdays uses ~3-5 GB of RAM. For a quick
# first pass, uncomment the next line to validate on a few recdays before
# expanding.
# mouse_recdays = mouse_recdays[:5]

from glm_analysis_v2 import build_data_dic_from_pfc
data_dic = build_data_dic_from_pfc(DATA_FOLDER, mouse_recdays)
mouse_recdays = sorted(data_dic.keys())   # in case any were skipped
print(f'\n{len(mouse_recdays)} recdays loaded')

  ab03_01092023_02092023: 7 sessions
  ab03_05092023_06092023: 8 sessions
  ab03_29082023_30082023: 9 sessions
  ah03_12082021_13082021: 8 sessions
  ah03_18082021_19082021: 8 sessions
  ah04_01122021_02122021: 8 sessions
  ah04_05122021_06122021: 8 sessions
  ah04_07122021_08122021: 8 sessions
  ah04_09122021_10122021: 8 sessions
  ah04_14122021_16122021: 8 sessions
  ah07_01092023_02092023: 7 sessions
  ah07_27082023_28082023: 7 sessions
  ah07_29082023_30082023: 7 sessions
  me08_06092021_09092021: 8 sessions
  me08_10092021_11092021: 6 sessions
  me08_12092021_13092021: 7 sessions
  me10_09122021_10122021: 9 sessions
  me10_14122021_15122021: 6 sessions
  me10_17122021_19122021: 8 sessions
  me10_20122021_21122021: 6 sessions
  me11_01122021_02122021: 8 sessions
  me11_05122021_06122021: 9 sessions
  me11_07122021_08122021: 8 sessions
  me11_09122021_10122021: 9 sessions
  me11_12122021_13122021: 9 sessions

25 recdays loaded


In [5]:
# 8 PFC regressors (HD dropped). Pass this list to every run_glm_analysis,
# compute_tuning_arrays, and plot_* call so the index maps match the design
# matrix throughout.
PFC_REGRESSORS = ['place', 'goal_progress', 'speed', 'acceleration',
                  'time_since_reward', 'time_to_reward',
                  'distance_since_reward', 'distance_to_reward']
print('PFC regressors:', PFC_REGRESSORS)

PFC regressors: ['place', 'goal_progress', 'speed', 'acceleration', 'time_since_reward', 'time_to_reward', 'distance_since_reward', 'distance_to_reward']


## Baseline GLM fit + per-regressor tuning

In [ ]:
reload(glm_analysis_v2)
from glm_analysis_v2 import (
    run_glm_analysis, compute_tuning_arrays, group_tuning_by_mouse,
    plot_tuning_piecharts_binary,
)

GLM_results, Permutation_results = run_glm_analysis(
    mouse_recdays, data_dic,
    regressors_to_include=PFC_REGRESSORS,
)

Processing recording days:   0%|          | 0/25 [00:00<?, ?it/s]


ab03_01092023_02092023


/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/code/glm_analysis_v2.py:748: RuntimeWarning: invalid value encountered in cast
  hd_bin_idx = np.clip(np.floor((hd_all % 360) / 10).astype(int), 0, 35)


  Design matrix: 26462 rows × 91 cols (regressors: ['place', 'goal_progress', 'speed', 'acceleration', 'time_from_reward', 'time_to_reward', 'distance_from_reward', 'distance_to_reward'], parameterization=all_bins)


In [ ]:
tuned_dict = compute_tuning_arrays(
    GLM_results, Permutation_results,
    regressors_to_include=PFC_REGRESSORS,
)
mouse_tuning_concat = group_tuning_by_mouse(tuned_dict)
plot_tuning_piecharts_binary(mouse_tuning_concat,
                             regressors_to_include=PFC_REGRESSORS)

## Sanity check: decile distributions

Pools the same values that `run_glm_analysis` feeds to `compute_decile_edges`
(after the `Locs <= 21` filter, after downsampling), and overlays the chosen
decile boundaries plus the 1st/99th-percentile clip lines. Quick visual
confirmation that each variable's bins land where you'd expect — e.g. that
`time_from_reward` isn't so spiky at 0 that several deciles collapse.

In [ ]:
from glm_analysis_v2 import plot_decile_distributions
pooled = plot_decile_distributions(mouse_recdays, data_dic)

## No-lookahead variant

Drops `time_to_reward` and `distance_to_reward`. Tests whether the
goal-progress signal survives without forward-looking variables (whose
information about the upcoming reward is partly tautological with goal
progress).

In [ ]:
PFC_REGRESSORS_NL = ['place', 'goal_progress', 'speed', 'acceleration',
                     'time_since_reward', 'distance_since_reward']

GLM_results_nl, Permutation_results_nl = run_glm_analysis(
    mouse_recdays, data_dic,
    regressors_to_include=PFC_REGRESSORS_NL,
)
tuned_dict_nl = compute_tuning_arrays(
    GLM_results_nl, Permutation_results_nl,
    regressors_to_include=PFC_REGRESSORS_NL,
)
mouse_tuning_concat_nl = group_tuning_by_mouse(tuned_dict_nl)
plot_tuning_piecharts_binary(mouse_tuning_concat_nl,
                             regressors_to_include=PFC_REGRESSORS_NL)

## Extended analysis: CPD + joint `time_any` group

Re-fits with `compute_cpd=True` and a joint test of "any time information"
(`time_since` + `time_to` dropped together). Produces the headline GP-vs-time
comparison: simplified pies, time-vs-progress overlap stacked bars, per-neuron
CPD scatter.

In [ ]:
reload(glm_analysis_v2)
from glm_analysis_v2 import (
    run_glm_analysis, plot_time_vs_progress_overlap, plot_cpd_time_vs_progress,
)

GLM_results_ext, Permutation_results_ext, CPD_results_ext = run_glm_analysis(
    mouse_recdays, data_dic,
    regressors_to_include=PFC_REGRESSORS,
    compute_cpd=True,
    joint_drop_groups=[('time_any', ['time_since_reward', 'time_to_reward'])],
)
print('Done. CPD and joint time_any stored.')

In [ ]:
tuned_dict_ext = compute_tuning_arrays(
    GLM_results_ext, Permutation_results_ext,
    regressors_to_include=PFC_REGRESSORS,
)
mouse_tuning_concat_ext = group_tuning_by_mouse(tuned_dict_ext)
plot_tuning_piecharts_binary(mouse_tuning_concat_ext,
                             regressors_to_include=PFC_REGRESSORS)

In [ ]:
plot_time_vs_progress_overlap(mouse_tuning_concat_ext,
                              regressors_to_include=PFC_REGRESSORS,
                              use_joint_time=False)

In [ ]:
cpd_gp, cpd_time = plot_cpd_time_vs_progress(CPD_results_ext, group_by_mouse=True)

by_mouse_diff = defaultdict(list)
for mr, neuron_dict in CPD_results_ext.items():
    mouse = mr.split('_')[0]
    for cpd in neuron_dict.values():
        g = cpd.get('goal_progress', np.nan)
        t = cpd.get('time_any', np.nan)
        if not (np.isnan(g) or np.isnan(t)):
            by_mouse_diff[mouse].append(t - g)

print('Per-mouse mean Δ (CPD_time_any − CPD_progress):')
for mouse in sorted(by_mouse_diff.keys()):
    diffs = np.array(by_mouse_diff[mouse])
    print(f'  {mouse}: μ={diffs.mean():+.5f}  n={len(diffs)}')

## Firming up: full-model R², normalized CPD, column-matched

Anchors CPD magnitudes with model-fit quality and runs a column-matched
single-regressor comparison (GP 10 cols vs `time_since_reward` 10 cols).

In [ ]:
from glm_analysis_v2 import plot_full_model_r2
plot_full_model_r2(CPD_results_ext)

In [ ]:
plot_cpd_time_vs_progress(CPD_results_ext, normalize='r2_full')

In [ ]:
# Column-matched: GP(10) vs time_since(10) — single-regressor, NOT joint.
gp_cpd, ts_cpd, mouse_tag = [], [], []
for mr, neuron_dict in CPD_results_ext.items():
    mouse = mr.split('_')[0]
    for cpd in neuron_dict.values():
        g = cpd.get('goal_progress', np.nan)
        t = cpd.get('time_from_reward', np.nan)  # canonical = time_since_reward
        if np.isfinite(g) and np.isfinite(t):
            gp_cpd.append(g); ts_cpd.append(t); mouse_tag.append(mouse)
gp_cpd = np.array(gp_cpd); ts_cpd = np.array(ts_cpd); mouse_tag = np.array(mouse_tag)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
ax = axes[0]
for i, mouse in enumerate(sorted(np.unique(mouse_tag))):
    msk = mouse_tag == mouse
    ax.scatter(gp_cpd[msk], ts_cpd[msk], alpha=0.6, s=18,
               color=plt.get_cmap('tab10')(i % 10), label=mouse)
lim = max(0.001, max(gp_cpd.max(), ts_cpd.max()) * 1.05)
ax.plot([0, lim], [0, lim], 'k--', lw=1)
ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.set_aspect('equal')
ax.set_xlabel('CPD goal progress (10 cols)')
ax.set_ylabel('CPD time since reward (10 cols)')
ax.set_title('Column-matched: GP vs time-since')
ax.legend(fontsize=8, title='Mouse')

ax = axes[1]
d = ts_cpd - gp_cpd
ax.hist(d, bins=40, color='darkorange', alpha=0.7, edgecolor='black')
ax.axvline(0, color='black', ls='--', lw=1.5)
ax.axvline(d.mean(), color='red', lw=2, label=f'Mean Δ = {d.mean():+.5f}')
tval, pval = st.ttest_1samp(d, 0)
ax.text(0.05, 0.95, f'n={len(d)}\nt={tval:.2f}\np={pval:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('CPD_time_since − CPD_progress')
ax.set_ylabel('Number of neurons')
ax.set_title('Δ (column-matched)')
ax.legend()
plt.suptitle('PFC: column-matched time-since vs goal-progress CPD', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## Raised-cosine basis variant (robustness check)

Refits with smooth raised-cosine bumps in place of hard decile binning for the
continuous regressors. If the time-vs-progress conclusion survives, it isn't
an artefact of the hard-binning choice.

In [ ]:
GLM_results_rc, Permutation_results_rc, CPD_results_rc = run_glm_analysis(
    mouse_recdays, data_dic,
    regressors_to_include=PFC_REGRESSORS,
    compute_cpd=True,
    joint_drop_groups=[('time_any', ['time_since_reward', 'time_to_reward'])],
    continuous_basis='raised_cosine',
)
tuned_dict_rc = compute_tuning_arrays(
    GLM_results_rc, Permutation_results_rc,
    regressors_to_include=PFC_REGRESSORS,
)
mouse_tuning_concat_rc = group_tuning_by_mouse(tuned_dict_rc)
plot_tuning_piecharts_binary(mouse_tuning_concat_rc,
                             regressors_to_include=PFC_REGRESSORS)
plot_time_vs_progress_overlap(mouse_tuning_concat_rc,
                              regressors_to_include=PFC_REGRESSORS)
plot_cpd_time_vs_progress(CPD_results_rc)

## 20-bin GP column-matched (GP=20 vs joint `time_any`=20, `distance_any`=20)

`goal_progress` re-binned to 20 cols so it matches both joint 20-col groups.
Removes the column-count advantage that `time_any` / `distance_any` had over
the 10-bin GP in the extended section above. Design matrix grows from 91 to
101 cols.

In [ ]:
GLM_results_gp20, Permutation_results_gp20, CPD_results_gp20 = run_glm_analysis(
    mouse_recdays, data_dic,
    regressors_to_include=PFC_REGRESSORS,
    gp_n_bins=20,
    compute_cpd=True,
    joint_drop_groups=[
        ('time_any',     ['time_since_reward', 'time_to_reward']),
        ('distance_any', ['distance_since_reward', 'distance_to_reward']),
    ],
)
tuned_dict_gp20 = compute_tuning_arrays(
    GLM_results_gp20, Permutation_results_gp20,
    regressors_to_include=PFC_REGRESSORS, gp_n_bins=20,
)
mouse_tuning_concat_gp20 = group_tuning_by_mouse(tuned_dict_gp20)
plot_tuning_piecharts_binary(mouse_tuning_concat_gp20,
                             regressors_to_include=PFC_REGRESSORS)
plot_time_vs_progress_overlap(mouse_tuning_concat_gp20,
                              regressors_to_include=PFC_REGRESSORS)
plot_cpd_time_vs_progress(CPD_results_gp20, group_by_mouse=True)

In [ ]:
# Column-matched distance: GP(20 cols) vs distance_any(20 cols)
gp_cpd, da_cpd, mouse_tag = [], [], []
for mr, neuron_dict in CPD_results_gp20.items():
    mouse = mr.split('_')[0]
    for cpd in neuron_dict.values():
        g = cpd.get('goal_progress', np.nan)
        d = cpd.get('distance_any', np.nan)
        if np.isfinite(g) and np.isfinite(d):
            gp_cpd.append(g); da_cpd.append(d); mouse_tag.append(mouse)
gp_cpd = np.array(gp_cpd); da_cpd = np.array(da_cpd); mouse_tag = np.array(mouse_tag)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
ax = axes[0]
for i, mouse in enumerate(sorted(np.unique(mouse_tag))):
    msk = mouse_tag == mouse
    ax.scatter(gp_cpd[msk], da_cpd[msk], alpha=0.6, s=18,
               color=plt.get_cmap('tab10')(i % 10), label=mouse)
lim = max(0.001, max(gp_cpd.max(), da_cpd.max()) * 1.05)
ax.plot([0, lim], [0, lim], 'k--', lw=1)
ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.set_aspect('equal')
ax.set_xlabel('CPD goal progress (20 cols)')
ax.set_ylabel('CPD distance (any) (20 cols)')
ax.set_title('Column-matched: GP vs distance')
ax.legend(fontsize=8, title='Mouse')

ax = axes[1]
d = da_cpd - gp_cpd
ax.hist(d, bins=40, color='seagreen', alpha=0.7, edgecolor='black')
ax.axvline(0, color='black', ls='--', lw=1.5)
ax.axvline(d.mean(), color='red', lw=2, label=f'Mean Δ = {d.mean():+.5f}')
tval, pval = st.ttest_1samp(d, 0)
ax.text(0.05, 0.95, f'n={len(d)}\nt={tval:.2f}\np={pval:.2e}',
        transform=ax.transAxes, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel('CPD_distance − CPD_progress')
ax.set_ylabel('Number of neurons')
ax.set_title('Δ (column-matched)')
ax.legend()
plt.suptitle('PFC: column-matched distance vs goal-progress CPD (20 vs 20)', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## Reference-coded sanity check

Re-fits one recday under the textbook reference coding (intercept + drop first
bin of each block) and asserts that R², CPD, and ΔR² match the default
no-intercept all-bins fit to numerical precision. F-stats differ by a constant
df-scaling but the substantive significance flags agree.

In [ ]:
mr_check = mouse_recdays[0]
data_check = {mr_check: data_dic[mr_check]}
np.random.seed(42)
GA, PA, CA = run_glm_analysis(
    [mr_check], data_check, regressors_to_include=PFC_REGRESSORS,
    compute_cpd=True,
    joint_drop_groups=[('time_any', ['time_since_reward', 'time_to_reward'])],
)
np.random.seed(42)
GR, PR, CR = run_glm_analysis(
    [mr_check], data_check, regressors_to_include=PFC_REGRESSORS,
    compute_cpd=True,
    joint_drop_groups=[('time_any', ['time_since_reward', 'time_to_reward'])],
    parameterization='reference_coded',
)

nids = sorted(set(CA[mr_check].keys()) & set(CR[mr_check].keys()))
r2_diffs, cpd_diffs, dr2_diffs = [], {}, {}
for nid in nids:
    a, r = CA[mr_check][nid], CR[mr_check][nid]
    r2_diffs.append(abs(a['__r2_full__'] - r['__r2_full__']))
    for k in a:
        if k.startswith('__') or k not in r: continue
        cpd_diffs.setdefault(k, []).append(abs(a[k] - r[k]))
        dr2_diffs.setdefault(k, []).append(abs(a['__delta_r2__'][k] - r['__delta_r2__'][k]))

print(f'Comparing {len(nids)} neurons in {mr_check}')
print(f'max |Δ r2_full|       = {max(r2_diffs):.2e}')
print(f'max |Δ CPD| (any reg) = {max(max(v) for v in cpd_diffs.values()):.2e}')
print(f'max |Δ ΔR²| (any reg) = {max(max(v) for v in dr2_diffs.values()):.2e}')
assert max(r2_diffs) < 1e-8
assert max(max(v) for v in cpd_diffs.values()) < 1e-8
assert max(max(v) for v in dr2_diffs.values()) < 1e-8

ta = compute_tuning_arrays(GA, PA, regressors_to_include=PFC_REGRESSORS)
tr = compute_tuning_arrays(GR, PR, regressors_to_include=PFC_REGRESSORS,
                           parameterization='reference_coded')
agree = int(np.sum(ta[mr_check] == tr[mr_check]))
total = int(ta[mr_check].size)
print(f'tuning-flag agreement: {agree}/{total}  ({100*agree/total:.1f}%)')
print('PFC sanity check PASSED — parameterization-invariant.')